In [11]:
import sys
import os

# ensure we're always running from project root
os.chdir(os.path.dirname(os.getcwd()))
sys.path.insert(0, '.')

In [17]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])

CompletedProcess(args=['c:\\Development\\github\\retail-analysis\\venv\\Scripts\\python.exe', '-m', 'pip', 'install', '-r', 'requirements.txt'], returncode=0)

In [18]:
from src.load import load_data
from src.clean import clean_data

df_raw = load_data()
df = clean_data(df_raw)

── Data loaded ──────────────────────────
Rows:              1,033,019
Date range:        2009-12-01 → 2011-12-09
Unique customers:  5,940
Cancellations:     19,100
Null Customer IDs: 235,150
─────────────────────────────────────────
── Cleaning summary ─────────────────────
Rows before: 1,033,019
Rows after:  1,021,374
Removed:     11,645
Outlier rows flagged: 7
─────────────────────────────────────────


In [19]:
cancellations = df[df['Invoice'].str.startswith('C', na=False)]

jan_cancellations = cancellations[cancellations['InvoiceDate'].dt.strftime('%Y-%m') == '2011-01']

In [20]:
print(f"Number of cancellation invoices: {jan_cancellations['Invoice'].nunique()}")
print(f"\nTop cancelled products:")
print(jan_cancellations.groupby('Description')['TotalPrice'].sum().sort_values().head(10))
print(f"\nTop cancellation invoices by value:")
print(jan_cancellations.groupby('Invoice')['TotalPrice'].sum().sort_values().head(5))

Number of cancellation invoices: 247

Top cancelled products:
Description
MEDIUM CERAMIC TOP STORAGE JAR        -77183.60
REGENCY CAKESTAND 3 TIER               -1058.10
SET/4 WHITE RETRO STORAGE CUBES         -838.80
VINTAGE RED KITCHEN CABINET             -515.50
BLUE CHARLIE+LOLA PERSONAL DOORSIGN     -442.50
ASSORTED COLOURS SILK FAN               -390.00
CHILLI LIGHTS                           -310.25
RED RETROSPOT CAKE STAND                -284.70
TWO DOOR CURIO CABINET                  -209.85
OCEAN SCENT CANDLE IN JEWELLED BOX      -205.74
Name: TotalPrice, dtype: float64

Top cancellation invoices by value:
Invoice
C541433   -77183.60
C540417    -1520.11
C542426    -1241.43
C540307     -811.86
C542138     -563.15
Name: TotalPrice, dtype: float64


Invoice C541433 accounts for £77,183

## Investigation

ID the customer and details of the invoice

In [23]:
print(df[df['Invoice'] == 'C541433'][['Invoice', 'StockCode', 'Description', 'Quantity', 'Price', 'TotalPrice', 'Customer ID', 'Country']])

        Invoice StockCode                     Description  Quantity  Price  \
550521  C541433     23166  MEDIUM CERAMIC TOP STORAGE JAR    -74215   1.04   

        TotalPrice  Customer ID         Country  
550521    -77183.6        12346  United Kingdom  


Find the original invoice for this customer around the same time

In [ ]:
customer = df[df['Invoice'] == 'C541433']['Customer ID'].iloc[0]
print(f"Customer ID: {customer}")

print(df[df['Customer ID'] == customer].sort_values('InvoiceDate')[['Invoice', 'Description', 'Quantity', 'TotalPrice', 'InvoiceDate', 'Country']])

Customer ID: 12346
        Invoice                         Description  Quantity  TotalPrice  \
105058   499763          RED SPOTTY CHILDS UMBRELLA         1        3.25   
105059   499763               EDWARDIAN PARASOL RED         1        5.95   
105060   499763           EDWARDIAN PARASOL NATURAL         1        5.95   
105061   499763             EDWARDIAN PARASOL BLACK         1        5.95   
105062   499763              EDWARDIAN PARASOL PINK         1        5.95   
246330   513774   DOORMAT UNION JACK GUNS AND ROSES         1        7.49   
246329   513774                    DOORMAT RED SPOT         1        7.49   
246328   513774                    DOORMAT AIRMAIL          1        7.49   
246327   513774                 DOORMAT NEW ENGLAND         1        7.49   
246326   513774             DOORMAT WELCOME PUPPIES         1        7.49   
246325   513774         DOORMAT PEACE ON EARTH BLUE         1        7.49   
246324   513774                  DOORMAT FAIRY CAKE      

Finding the info of the relevant invoices

In [25]:
print(df[df['Customer ID'] == 12346][['Invoice', 'Description', 'Quantity', 'Price', 'TotalPrice', 'InvoiceDate']].sort_values('InvoiceDate'))
print(f"\nTotal spent (net): £{df[df['Customer ID'] == 12346]['TotalPrice'].sum():,.2f}")

        Invoice                         Description  Quantity  Price  \
105058   499763          RED SPOTTY CHILDS UMBRELLA         1   3.25   
105059   499763               EDWARDIAN PARASOL RED         1   5.95   
105060   499763           EDWARDIAN PARASOL NATURAL         1   5.95   
105061   499763             EDWARDIAN PARASOL BLACK         1   5.95   
105062   499763              EDWARDIAN PARASOL PINK         1   5.95   
246330   513774   DOORMAT UNION JACK GUNS AND ROSES         1   7.49   
246329   513774                    DOORMAT RED SPOT         1   7.49   
246328   513774                    DOORMAT AIRMAIL          1   7.49   
246327   513774                 DOORMAT NEW ENGLAND         1   7.49   
246326   513774             DOORMAT WELCOME PUPPIES         1   7.49   
246325   513774         DOORMAT PEACE ON EARTH BLUE         1   7.49   
246324   513774                  DOORMAT FAIRY CAKE         1   7.49   
246323   513774           DOORMAT RESPECTABLE HOUSE         1   

Seeing if we still have the original invoice after data cleaning

In [26]:
# Find the original order that C541433 relates to
# C invoices typically reference the original by replacing C with blank
original_invoice = '541433'
print(df[df['Invoice'] == original_invoice][['Invoice', 'Description', 'Quantity', 'Price', 'TotalPrice', 'InvoiceDate']])

Empty DataFrame
Columns: [Invoice, Description, Quantity, Price, TotalPrice, InvoiceDate]
Index: []


**Summary**:
Customer 12346 is a retail buyer who normally spends £5-£6 per item in small orders. Sudenly they placed an order of 74215 items. It was then cancelled 16 mins later. This was obviously an error by the customer, entering an incorrect qty.

# Dec Investigation

In [33]:
dec_cancellations = cancellations[cancellations['InvoiceDate'].dt.strftime('%Y-%m') == '2011-12']

In [34]:
print(f"Number of cancellation invoices: {dec_cancellations['Invoice'].nunique()}")
print(f"\nTop cancelled products:")
print(dec_cancellations.groupby('Description')['TotalPrice'].sum().sort_values().head(10))
print(f"\nTop cancellation invoices by value:")
print(dec_cancellations.groupby('Invoice')['TotalPrice'].sum().sort_values().head(5))

Number of cancellation invoices: 138

Top cancelled products:
Description
PAPER CRAFT , LITTLE BIRDIE          -168469.60
CRUK Commission                        -1100.44
OFFICE MUG WARMER PINK                  -637.50
VINTAGE DOILY DELUXE SEWING KIT         -243.95
GUMBALL MAGAZINE RACK                   -169.65
VICTORIAN SEWING BOX LARGE              -109.50
COLOURED GLASS STAR T-LIGHT HOLDER      -106.65
REGENCY CAKESTAND 3 TIER                -103.95
BOTANICAL GARDENS WALL CLOCK             -95.80
SET OF 10 LED DOLLY LIGHTS               -93.86
Name: TotalPrice, dtype: float64

Top cancellation invoices by value:
Invoice
C581484   -168469.60
C580726     -1100.44
C580971      -664.86
C580954      -436.20
C580968      -381.95
Name: TotalPrice, dtype: float64


Invoice C581484 accounts for £168,469

In [35]:
customer = df[df['Invoice'] == 'C581484']['Customer ID'].iloc[0]
print(f"Customer ID: {customer}")

print(df[df['Customer ID'] == customer].sort_values('InvoiceDate')[['Invoice', 'Description', 'Quantity', 'TotalPrice', 'InvoiceDate', 'Country']])

Customer ID: 16446
         Invoice                  Description  Quantity  TotalPrice  \
680493    553573       PANTRY SCRUBBING BRUSH         1        1.65   
680494    553573          PANTRY PASTRY BRUSH         1        1.25   
1019901   581483  PAPER CRAFT , LITTLE BIRDIE     80995   168469.60   
1019902  C581484  PAPER CRAFT , LITTLE BIRDIE    -80995  -168469.60   

                InvoiceDate         Country  
680493  2011-05-18 09:52:00  United Kingdom  
680494  2011-05-18 09:52:00  United Kingdom  
1019901 2011-12-09 09:15:00  United Kingdom  
1019902 2011-12-09 09:27:00  United Kingdom  


This looks to be the same issue, the customer has 2 previous orders then a huge order of 80995. This was cancelled 12 mins after. Interestingly, the order was flagged as an outlier. We should think about changing the cleaning function to remove outlier cancelations and orders together.

In [36]:
print(df[df['Invoice'] == '581483'][['Invoice', 'Quantity', 'IsOutlier']])

        Invoice  Quantity  IsOutlier
1019901  581483     80995       True


Revenue peaks sharply in October–November each year, reaching ~2× a typical month, consistent with Christmas wholesale ordering patterns. Forecasting and stock planning should account for this concentration.

Two cancellation spikes in January 2011 (13.65%) and December 2011 (28.51%) are attributable to single anomalous transactions — likely data entry errors — rather than systemic operational issues. Excluding these two invoices, the underlying cancellation rate is approximately 2–3% and consistent across both years.

The headline cancellation rate of 7.2% overstates the operational problem. Excluding two data entry errors, the true underlying rate is 2.4% — representing approximately £470k in genuine lost revenue against £19.6m gross. This is within normal range for a gift retailer of this type.